# Module 2: Hooks (10 min)

Add a rate limiter hook to the customer service agent. Hooks let you inject deterministic code into the agent loop — before invocations, before/after tool calls — without changing the agent's logic.

**Prerequisites:** Module 1 completed, `strands-agents` installed

In [ ]:
!pip install -q -r requirements.txt

---

## Part 1: Build a Rate Limiter Hook

Hooks implement `HookProvider` and register callbacks for lifecycle events. This one caps each tool at 3 calls per request.

In [ ]:
from strands import Agent, tool
from strands.hooks import (
    HookProvider, HookRegistry,
    BeforeInvocationEvent, BeforeToolCallEvent,
)
from customer_service_tools import lookup_customer, get_order_history, process_refund


class RateLimiterHook(HookProvider):
    """Caps each tool at max_calls per agent invocation."""

    def __init__(self, max_calls: int = 3):
        self.max_calls = max_calls
        self.counts: dict[str, int] = {}

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self.reset)
        registry.add_callback(BeforeToolCallEvent, self.check)

    def reset(self, event: BeforeInvocationEvent) -> None:
        """Reset counts at the start of each invocation."""
        self.counts = {}
        print("[HOOK] 🔄 Rate limiter reset")

    def check(self, event: BeforeToolCallEvent) -> None:
        """Check and enforce the rate limit before each tool call."""
        name = event.tool_use["name"]
        self.counts[name] = self.counts.get(name, 0) + 1
        print(f"[HOOK] 📊 {name}: call {self.counts[name]}/{self.max_calls}")

        if self.counts[name] > self.max_calls:
            event.cancel_tool = (
                f"'{name}' hit the {self.max_calls}-call limit. "
                "Do NOT call this tool again."
            )
            print(f"[HOOK] 🚫 BLOCKED: {name} exceeded limit!")


print("✅ RateLimiterHook defined")

---

## Part 2: Attach the Hook to the Agent

Pass hooks via the `hooks` parameter. The agent loop will call your hook at the right lifecycle points.

In [ ]:
SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise. Use the available tools to look up customer
information and process requests."""

agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    hooks=[RateLimiterHook(max_calls=3)],
    system_prompt=SYSTEM_PROMPT,
)

# Normal request — should work fine
result = agent("I'm customer C-1001. What are my recent orders?")

---

## Part 3: Trigger the Rate Limit

Ask the agent something that would cause excessive tool calls. Watch the hook block it.

In [ ]:
# This prompt tries to get the agent to call lookup_customer many times
agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    hooks=[RateLimiterHook(max_calls=2)],  # Lower limit to trigger faster
    system_prompt=SYSTEM_PROMPT,
)

result = agent(
    "Look up customers C-1001, C-1002, C-1003, C-1004, and C-1005. "
    "Give me all their details."
)

---

## 🎯 Try It Yourself

Modify the hook to also track total calls across all tools (not just per-tool):

In [ ]:
# Challenge: Add a total_calls counter that blocks ALL tools after 5 total calls
# Hint: Add a self.total counter in reset() and check it in check()

# Your code here...

---

## What's Next

Hooks give you low-level control over the loop. In **Module 3: Skills + Steering**, you'll add workflow knowledge (skills) and business rule enforcement (steering handlers) to make the agent follow proper procedures.